# Kenya census retrieval and answer evaluation
CPU Colab notebook. Run from the top. You can upload census_pages.json from the first notebook or the original September 2023 PDF. Review the reference questions before inspecting search results.

In [ ]:
%pip -q install pypdf==5.9.0

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, io, json
from pypdf import PdfReader

upload = files.upload()
if len(upload) != 1:
    raise ValueError('Select one PDF or census_pages.json.')
name, raw = next(iter(upload.items()))
if name.lower().endswith('.pdf'):
    if hashlib.sha256(raw).hexdigest() != '49e15c1cc8d02f2cf8d62f5d9ef5983c6c2d3abd0db63619cb0f9409be6c25a2':
        raise ValueError('Use the September 2023 PDF from notebook 01.')
    pages = [{'pdf_page': i, 'text': p.extract_text() or ''} for i,p in enumerate(PdfReader(io.BytesIO(raw)).pages,1)]
else:
    pages = json.loads(raw)
if [p['pdf_page'] for p in pages] != list(range(1,88)) or '54%' not in pages[13]['text']:
    raise ValueError('Expected the 87-page census with the laboratory finding on PDF page 14.')
print('Loaded', len(pages), 'PDF pages.')

## Step 3 — Review and freeze the reference answers
Open each cited PDF page. These are seed pages, not an exhaustive list. Add any known equivalent pages before running the searches. The four unsupported cases test whether the assistant acknowledges the report’s scope.

In [ ]:
import csv
from pathlib import Path

out = Path('health_census_evaluation')
out.mkdir(exist_ok=True)
cases = [
    ('Q01','Among facilities with a laboratory, what share offered basic laboratory services?', '87% of facilities with a laboratory.', [14]),
    ('Q02','Among facilities with a laboratory, what share offered comprehensive laboratory services?', '54% of facilities with a laboratory.', [14]),
    ('Q03','How many facilities were targeted by the national census?', '14,366 facilities.', [12,20]),
    ('Q04','How many health facilities denied the census team access?', '203 facilities.', [20]),
    ('Q05','Which three datasets were merged for each facility?', 'Administrative, clinical and specialized datasets.', [19]),
    ('Q06','What was the national density of primary health care facilities?', '2.4 facilities per 10,000 population.', [15]),
    ('Q07','What average availability did the pharmacy section report for tracer drugs in the 2023 census?', '62% average availability of tracer drugs.', [37]),
    ('Q08','Which comprehensive maternity equipment tracer relates to obtaining blood?', 'Access to a blood bank.', [45]),
    ('Q09','Which facilities have pregnancy test kits in stock today?', 'Current stock cannot be established.', []),
    ('Q10','Which hospital will run out of oxygen next week?', 'The report cannot forecast a facility stockout.', []),
    ('Q11','What is the name of the nurse on duty tonight at a particular maternity ward?', 'Current staff rosters are outside the report.', []),
    ('Q12','What is the confirmed national tracer drug availability for 2026?', 'The 2023 report cannot establish the 2026 value.', []),
]
def write_csv(name, header, rows):
    with (out/name).open('w',newline='',encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)
write_csv('reference_answers.csv',['id','question','expected','seed_pdf_pages'],cases)
for ident,question,answer,refs in cases:
    print(ident, question, '|', answer, '|', refs)

## Step 4 — Run the keyword baseline
Each search ranks PDF pages. Hit@3 means at least one reviewed reference page appears in the first three. Unsupported questions have no expected page.

In [ ]:
import math, re
from collections import Counter

texts = [' '.join(p['text'].split()) for p in pages]
questions = [c[1] for c in cases]
def tokens(text):
    return re.findall(r'[a-z0-9]+',text.lower())
counts = [Counter(tokens(text)) for text in texts]
df = Counter(term for page in counts for term in page)
def weighted_vector(counts):
    vec = {term: (1+math.log(freq))*(1+math.log((1+len(pages))/(1+df[term])))
           for term,freq in counts.items() if term in df}
    norm = math.sqrt(sum(value**2 for value in vec.values())) or 1
    return {term:value/norm for term,value in vec.items()}
page_vectors = [weighted_vector(count) for count in counts]
scores = []
for question in questions:
    query = weighted_vector(Counter(tokens(question)))
    scores.append([sum(weight * page.get(term,0) for term,weight in query.items())
                   for page in page_vectors])
rankings = {}
def evaluate(method, scores):
    top = [[i+1 for i in sorted(range(len(row)),key=lambda i: -row[i])[:3]] for row in scores]
    rankings[method] = top
    rows = []
    hits = 0
    for (ident,_,_,refs), found in zip(cases,top):
        hit = bool(set(found).intersection(refs)) if refs else None
        hits += int(hit is True)
        rows.append((ident,found,hit))
    write_csv(method+'.csv',['id','retrieved_pdf_pages','seed_page_hit'],rows)
    print(method, 'seed-page hits:', hits, '/', sum(bool(c[3]) for c in cases))
evaluate('keyword', scores)

## Step 5 — Compare semantic search on the same questions
This downloads an open embedding model on first use. Split long pages into overlapping passages to limit truncation. Each page receives its best passage score.

In [ ]:
%pip -q install sentence-transformers

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device='cpu')
passages, owners = [], []
for i,text in enumerate(texts):
    words = text.split()
    for start in range(0,len(words),70):
        passages.append(' '.join(words[start:start+90]))
        owners.append(i)
vectors = model.encode(passages,normalize_embeddings=True,show_progress_bar=True)
query_vectors = model.encode(questions,normalize_embeddings=True)
similarity = query_vectors @ vectors.T
page_scores = np.full((len(questions),len(pages)), -np.inf)
for j,i in enumerate(owners):
    page_scores[:,i] = np.maximum(page_scores[:,i],similarity[:,j])
evaluate('semantic', page_scores)

## Step 6 — Prepare the answer review
Use the semantic search results if available, otherwise the keyword results. For each case, give your local LLM only the exported prompt. Keep the expected answer hidden from the model. In the review CSV record the actual response, whether the fact and denominator are correct, whether the cited page supports it, and whether it appropriately declines an unsupported question. Manually inspect seed-page misses for alternative valid pages and report these separately. Keep raw rankings.

In [ ]:
import shutil
method = 'semantic' if 'semantic' in rankings else 'keyword'
prompts = []
for case, found in zip(cases,rankings[method]):
    ident, question, _, _ = case
    evidence = '\n\n'.join(f'PDF page {i}: {texts[i-1]}' for i in found)
    prompts.append({'id':ident,'method':method,'prompt':f'Use only the supplied pages. Cite PDF page numbers. Preserve denominators and dates. If the pages cannot answer, say so.\nQuestion: {question}\n{evidence}'})
(out/'llm_prompts.json').write_text(json.dumps(prompts,indent=2),encoding='utf-8')
write_csv('answer_review.csv',['id','actual_response','fact_and_denominator_correct','citation_supported','appropriate_abstention','notes'],[(c[0],'','','','','') for c in cases])
archive = shutil.make_archive('health_census_evaluation','zip',out)
files.download(archive)